# Demo: Von der offenen Exploration mit PyGWalker zur fokussierten Dash-App

## Leitfrage

> **Wann treten in einem Restaurantdatensatz hohe Trinkgelder auf, und wie unterscheiden sich Wochentage, Tageszeiten und Gruppengrößen?**

Wir verwenden die in Plotly enthaltenen **Tips-Daten**. PyGWalker unterstützt zunächst die offene, visuelle Exploration. Anschließend überführen wir ausgewählte Fragen in eine kleine Dash-App mit klar definierten Filtern und koordinierten Ansichten.

## 0. Vorbereitung

Die Installationszelle ist nur erforderlich, wenn ein Import fehlschlägt. PyGWalker benötigt eine aktuelle Python-Version; für dieses Notebook wird Python 3.10 oder neuer empfohlen.

In [ ]:
# Bei Bedarf das Kommentarzeichen entfernen und die Zelle einmal ausführen:
#%pip install plotly pandas dash pygwalker runcell

In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html
from IPython.display import display
import runcell

---
## 1. Daten und Analysefrage kennenlernen

Jede Zeile beschreibt eine Restaurantrechnung. Der Datensatz enthält den Rechnungsbetrag (`total_bill`), das Trinkgeld (`tip`), Angaben zu Tag und Tageszeit sowie die Gruppengröße (`size`).

Wir ergänzen den prozentualen Trinkgeldanteil `tip_pct`. Er erlaubt einen faireren Vergleich unterschiedlich hoher Rechnungen als der absolute Trinkgeldbetrag allein.

In [ ]:
tips = px.data.tips().copy()
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
tips["day"] = pd.Categorical(
    tips["day"], categories=["Thur", "Fri", "Sat", "Sun"], ordered=True
)

required = {"total_bill", "tip", "sex", "smoker", "day", "time", "size", "tip_pct"}
assert required.issubset(tips.columns)
assert tips["total_bill"].gt(0).all()

print(f"Beobachtungen: {len(tips)}")
print(f"Fehlende Werte: {int(tips.isna().sum().sum())}")
tips.head()

### Explorationsfragen

1. Steigt das absolute Trinkgeld mit dem Rechnungsbetrag?
2. Bleibt der prozentuale Trinkgeldanteil dabei konstant?
3. Unterscheiden sich Lunch und Dinner?
4. Gibt es Unterschiede nach Wochentag oder Gruppengröße?
5. Welche Beobachtungen fallen als Ausnahmen auf?

Diese Fragen sind bewusst offen. Wir wissen noch nicht, welche davon später in eine Dash-App gehören.

In [ ]:
preview = px.scatter(
    tips, x="total_bill", y="tip_pct", color="time", symbol="time",
    hover_data={"tip": ":.2f", "size": True, "day": True, "smoker": True},
    labels={
        "total_bill": "Rechnungsbetrag", "tip_pct": "Trinkgeldanteil (%)",
        "time": "Tageszeit", "tip": "Trinkgeld", "size": "Gruppengröße",
        "day": "Tag", "smoker": "Raucherbereich",
    },
    title="Erste Orientierung: Rechnungsbetrag und Trinkgeldanteil",
)
preview.update_layout(template="plotly_white")
preview.show(config={"displaylogo": False})

---
## 2. Offene Exploration mit PyGWalker

PyGWalker verwandelt einen DataFrame in eine visuelle Explorationsoberfläche. Felder lassen sich per Drag-and-drop auf Achsen, Farbe, Größe, Zeilen, Spalten und Filter ziehen.

Für diese Demo verwenden wir das lokale Browser-Backend und blenden das Cloud-Werkzeug aus. Die erste Anzeige kann etwas dauern.

In [ ]:
import pygwalker as pyg

In [ ]:
walker = pyg.walk(
    tips,
    computation="browser",
    default_tab="vis",
    appearance="light",
    show_cloud_tool=False,
)
walker

### Gemeinsame PyGWalker-Erkundung

Führen Sie die folgenden Schritte nacheinander aus und formulieren Sie nach jedem Schritt eine Beobachtung:

1. Ziehen Sie `total_bill` auf die X-Achse und `tip_pct` auf die Y-Achse.
2. Legen Sie `time` auf Farbe.
3. Legen Sie `size` auf Größe oder in einen Filter.
4. Facettieren Sie nach `day`, beispielsweise über Spalten.
5. Filtern Sie testweise auf Lunch oder Dinner.
6. Wechseln Sie vom Scatterplot zu einem Boxplot für `tip_pct` nach `day`.

**Diskussionsfrage:** Welche Interaktionen helfen bei der Exploration, wären aber für eine regelmäßig genutzte App unnötig oder verwirrend?

### Beobachtungen reproduzierbar prüfen

Visuelle Exploration erzeugt Hypothesen. Bevor wir daraus eine App bauen, prüfen wir ausgewählte Zusammenfassungen mit pandas. Die kleinen Gruppen und der beobachtende Datensatz erlauben keine kausalen Schlussfolgerungen.

In [ ]:
summary = (
    tips.groupby(["day", "time"], observed=True)
    .agg(
        observations=("tip_pct", "size"),
        median_bill=("total_bill", "median"),
        median_tip_pct=("tip_pct", "median"),
    )
    .reset_index()
)
summary.style.format({
    "median_bill": "{:.2f}",
    "median_tip_pct": "{:.1f}%",
})

---
## 3. Vom Explorationswerkzeug zur Daten-App

PyGWalker und Dash lösen unterschiedliche Aufgaben:

| Frage | PyGWalker | Plotly Dash |
|---|---|---|
| Hauptzweck | offene Exploration | gezielte, wiederholbare Anwendung |
| Zielgruppe | analysierende Person | definierte Nutzergruppe |
| Bedienung | viele frei kombinierbare Felder | ausgewählte Steuerelemente |
| Entwicklungsaufwand | gering für erste Analysen | höher, weil Layout und Logik programmiert werden |
| Reproduzierbarkeit | Konfiguration muss bewusst gespeichert werden | Verhalten ist im Code festgelegt |
| Geeignet, wenn … | die relevanten Fragen noch offen sind | Aufgaben und Arbeitsablauf bekannt sind |

Für die Dash-Demo reduzieren wir die offene Exploration auf drei Filter: **Tag**, **Tageszeit** und **Gruppengröße**. Zwei koordinierte Grafiken zeigen Zusammenhang und Verteilung.

---
## 4. Das mentale Modell von Dash

Eine einfache Dash-App besteht aus drei Teilen:

1. **Layout:** Welche Komponenten sind sichtbar?
2. **Input:** Welche Eigenschaften der Steuerelemente lösen eine Aktualisierung aus?
3. **Output:** Welche Eigenschaften der Zielkomponenten werden ersetzt?

```text
Dropdown / Checklist / Slider
            │
            ▼
        Callback
       filtert Daten
       und erzeugt Ausgaben
            │
            ▼
Scatterplot / Histogramm / Textzusammenfassung
```

Die Callback-Funktion bleibt eine normale Python-Funktion. Der Dekorator verbindet ihre Argumente und Rückgabewerte mit den Eigenschaften der Komponenten.

## 5. Die Darstellungslogik zunächst als normale Funktionen entwickeln

Wir trennen Datenfilterung und Figure-Erzeugung von Dash. Dadurch lassen sich beide Teile unabhängig lesen und testen.

In [ ]:
# Konstanten für die Dash-App
DAY_ORDER = ["Thur", "Fri", "Sat", "Sun"]
TIME_ORDER = ["Lunch", "Dinner"]
SAFE_COLORS = {"Lunch": "#4477AA", "Dinner": "#CC6677"}

# Funktionen für die Dash-App
def filter_tips(day, times, size_range):
    selected_times = TIME_ORDER if times is None else times
    lower, upper = size_range or [int(tips["size"].min()), int(tips["size"].max())]

    mask = tips["time"].isin(selected_times) & tips["size"].between(lower, upper)
    if day != "Alle Tage":
        mask &= tips["day"].astype(str).eq(day)
    return tips.loc[mask].copy()

def make_scatter(data):
    fig = px.scatter(
        data, x="total_bill", y="tip_pct", color="time", symbol="time",
        color_discrete_map=SAFE_COLORS,
        hover_data={"tip": ":.2f", "size": True, "day": True, "smoker": True},
        category_orders={"time": TIME_ORDER},
        labels={
            "total_bill": "Rechnungsbetrag", "tip_pct": "Trinkgeldanteil (%)",
            "time": "Tageszeit", "tip": "Trinkgeld", "size": "Gruppengröße",
            "day": "Tag", "smoker": "Raucherbereich",
        },
        title="Rechnungsbetrag und Trinkgeldanteil",
    )
    fig.update_traces(marker={"size": 10, "opacity": 0.78, "line": {"width": 0.6, "color": "white"}})
    fig.update_layout(template="plotly_white", legend_orientation="h", margin={"l": 60, "r": 20, "t": 70, "b": 60})
    fig.update_yaxes(range=[0, max(35, data["tip_pct"].max() * 1.08)] if len(data) else [0, 35])
    return fig

def make_distribution(data):
    fig = px.box(
        data, x="day", y="tip_pct", color="time", points="outliers",
        color_discrete_map=SAFE_COLORS, category_orders={"day": DAY_ORDER, "time": TIME_ORDER},
        labels={"day": "Tag", "tip_pct": "Trinkgeldanteil (%)", "time": "Tageszeit"},
        title="Verteilung des Trinkgeldanteils",
    )
    fig.update_layout(template="plotly_white", showlegend=False, margin={"l": 60, "r": 20, "t": 70, "b": 60})
    return fig

def make_summary(data):
    if data.empty:
        return "Keine Beobachtungen für die aktuelle Filterkombination."
    return (
        f"{len(data)} Beobachtungen · "
        f"Median der Rechnung: {data['total_bill'].median():.2f} · "
        f"Median des Trinkgeldanteils: {data['tip_pct'].median():.1f}%"
    )

In [ ]:
demo_subset = filter_tips("Sat", ["Dinner"], [2, 4])
assert not demo_subset.empty
assert demo_subset["day"].astype(str).eq("Sat").all()
assert demo_subset["time"].eq("Dinner").all()
print(make_summary(demo_subset))
make_scatter(demo_subset).show(config={"displaylogo": False})

---
## 6. Dash-Layout: Komponenten und IDs

Das Layout ist ein hierarchischer Baum aus HTML- und Dash-Komponenten. Jede Komponente, die ein Callback anspricht, erhält eine eindeutige `id`. Flexbox mit `flexWrap` ermöglicht ein einfaches responsives Verhalten.

In [ ]:
app = Dash(__name__)

# Dash control and graph styles
control_style = {"minWidth": "220px", "flex": "1 1 220px"}
graph_style = {"minWidth": "320px", "flex": "1 1 480px"}

# Dash app layout
app.layout = html.Div(
    [
        html.H1("Restaurantrechnungen und Trinkgeld"),
        html.P(
            "Die App untersucht Zusammenhänge und Verteilungen in einem historischen Demonstrationsdatensatz. "
            "Die Ergebnisse beschreiben Beobachtungen und erlauben keine kausalen Aussagen."
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Wochentag", htmlFor="day-filter"),
                        dcc.Dropdown(
                            id="day-filter",
                            options=[{"label": "Alle Tage", "value": "Alle Tage"}]
                            + [{"label": day, "value": day} for day in DAY_ORDER],
                            value="Alle Tage", clearable=False,
                        ),
                    ], style=control_style,
                ),
                html.Div(
                    [
                        html.Label("Tageszeit", htmlFor="time-filter"),
                        dcc.Checklist(
                            id="time-filter",
                            options=[{"label": value, "value": value} for value in TIME_ORDER],
                            value=TIME_ORDER, inline=True,
                        ),
                    ], style=control_style,
                ),
                html.Div(
                    [
                        html.Label("Gruppengröße", htmlFor="size-filter"),
                        dcc.RangeSlider(
                            id="size-filter", min=int(tips["size"].min()), max=int(tips["size"].max()),
                            step=1, value=[int(tips["size"].min()), int(tips["size"].max())],
                            marks={value: str(value) for value in range(int(tips["size"].min()), int(tips["size"].max()) + 1)},
                        ),
                    ], style=control_style,
                ),
            ],
            style={"display": "flex", "gap": "20px", "flexWrap": "wrap", "marginBottom": "20px"},
        ),
        html.Div(
            id="summary-text", role="status",
            style={"fontWeight": "600", "padding": "12px", "background": "#f3f5f7", "marginBottom": "12px"},
        ),
        html.Div(
            [
                html.Div(dcc.Graph(id="tip-scatter", responsive=True), style=graph_style),
                html.Div(dcc.Graph(id="tip-distribution", responsive=True), style=graph_style),
            ],
            style={"display": "flex", "gap": "16px", "flexWrap": "wrap"},
        ),
        html.Details(
            [
                html.Summary("Hinweise zur Interpretation"),
                html.Ul(
                    [
                        html.Li("Hover ergänzt Details, darf aber keine zentrale Aussage verstecken."),
                        html.Li("Kleine Teilgruppen führen zu unsicheren visuellen Vergleichen."),
                        html.Li("Die Daten zeigen Zusammenhänge, keine Ursachen."),
                    ]
                ),
            ]
        ),
    ],
    style={"fontFamily": "Arial, sans-serif", "maxWidth": "1400px", "margin": "0 auto", "padding": "20px",
           "backgroundColor": "#cbc3dfff", "color": "#111111", "minHeight": "100vh"},
)

---
## 7. Callback: drei Inputs, drei Outputs

Jede Änderung an **Dropdown**, **Checklist** oder **Range Slider** ruft die Funktion auf. Sie filtert den DataFrame einmal und erzeugt daraus beide Figuren sowie die Textzusammenfassung. So bleiben alle Ansichten im gleichen Filterzustand.

In [ ]:
# Die Callback-Funktion ist die Hauptfunktion des Dash-Apps.
@callback(
    Output("tip-scatter", "figure"),
    Output("tip-distribution", "figure"),
    Output("summary-text", "children"),
    Input("day-filter", "value"),
    Input("time-filter", "value"),
    Input("size-filter", "value"),
)

# Aktualisiert die Scatter- und Boxplots sowie die Zusammenfassung basierend auf den ausgewählten Filtern.
def update_app(day, times, size_range):
    filtered = filter_tips(day, times, size_range)
    return make_scatter(filtered), make_distribution(filtered), make_summary(filtered)

In [ ]:
# Die Callback-Funktion lässt sich auch ohne Browser testen.
test_scatter, test_distribution, test_summary = update_app("Sun", ["Dinner"], [2, 5])
assert len(test_scatter.data) > 0
assert len(test_distribution.data) > 0
assert "Beobachtungen" in test_summary
print(test_summary)

---
## 8. App im Notebook starten

Dash 2.11 oder neuer kann Apps direkt in Jupyter ausführen. `inline` zeigt die App unterhalb der Zelle. Falls der Port bereits belegt ist, wählen Sie beispielsweise `port=8060`.

Beenden Sie die App über die Stopp-Schaltfläche des Notebook-Kernels, bevor Sie die Startzelle erneut ausführen.

In [ ]:
app.run(jupyter_mode="inline", jupyter_height=850, debug=False)

---
## 9. Gestaltungsgrenzen, Accessibility und Responsiveness

| Risiko | Umsetzung in der Demo | Noch offen |
|---|---|---|
| Filterzustand wird übersehen | sichtbare Steuerelemente und aktualisierte Zusammenfassung | bei vielen Filtern zusätzlich aktive Filterchips oder Reset vorsehen |
| Farbe allein ist schwer unterscheidbar | Tageszeit wird über Farbe und Symbol kodiert | Kontraste mit realer Zielgruppe testen |
| Hover ist nicht für alle zugänglich | Kernaussage und Kennzahlen stehen außerhalb des Hovers | für Veröffentlichung zusätzlich zugängliche Datentabelle anbieten |
| Kleine Displays | `flexWrap`, Mindestbreiten und responsive Graphen | auf sehr kleinen Displays Grafiken besser untereinander anordnen |
| Leere Filterkombination | verständliche Textmeldung und leere Figures | Steuerelemente könnten ungültige Kombinationen proaktiv verhindern |
| Kleine Teilgruppen | Anzahl der Beobachtungen wird angezeigt | Unsicherheit wird noch nicht quantifiziert |

**Live-Test:** Browserfenster verkleinern, nur Lunch auswählen, Gruppengröße eins bis zwei wählen und anschließend einen einzelnen Wochentag aktivieren.

---
## 10. Abschluss: Werkzeugwahl als Teil des Analyseprozesses

1. **PyGWalker** half, mögliche Fragestellungen ohne vorab programmierte Oberfläche zu erkunden.
2. **pandas** prüfte ausgewählte Beobachtungen reproduzierbar.
3. **Dash** reduzierte den offenen Analysebereich auf einen verständlichen Arbeitsablauf.
4. Die Dash-App koordinierte Filter, zwei Visualisierungen und eine Textzusammenfassung.
5. Die Reduktion der Freiheitsgrade ist kein Nachteil, wenn Aufgaben und Zielgruppe bekannt sind.

### Reflexionsfragen

- Welche Erkenntnis aus PyGWalker wurde in die Dash-App übernommen?
- Welche PyGWalker-Freiheiten wurden bewusst nicht übernommen?
- Welche weitere Nutzeraufgabe würde einen neuen Filter oder eine neue Ansicht rechtfertigen?
- Wann wäre ein Notebook mit PyGWalker ausreichend, und wann lohnt sich eine Dash-App?
- Welche Ergänzung wäre für eine öffentlich zugängliche Anwendung zwingend?

---
## Weiterführende Dokumentation

- [Dash in Jupyter](https://dash.plotly.com/dash-in-jupyter)
- [Dash Callbacks](https://dash.plotly.com/basic-callbacks)
- [PyGWalker auf GitHub](https://github.com/Kanaries/pygwalker)